<a href="https://colab.research.google.com/github/Varshareddie25/ATM-JAVA-/blob/master/Task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 1 — Problem & Dataset

**Module:** 7006SCN Machine Learning and Big Data  
**Student:** VARSHAREDDY RAJIREDDYGARI

In [4]:
!git config --global user.name "Varsha Reddy"
!git config --global user.email "rvarshareddy.2003@gmail.com"

In [5]:
!git clone https://github.com/Varshareddie25/ATM-JAVA-.git

Cloning into 'ATM-JAVA-'...
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 9 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (9/9), 7.01 KiB | 2.33 MiB/s, done.


In [6]:
%cd ATM-JAVA-

/content/ATM-JAVA-/ATM-JAVA-


In [7]:
!git branch

* master


In [14]:
!git add .

In [15]:
!cp /content/*.ipynb .
!git add .

cp: cannot stat '/content/*.ipynb': No such file or directory


In [16]:
!git commit -m "Added Task 1 solution"

On branch master
Your branch is up to date with 'origin/master'.

nothing to commit, working tree clean


In [18]:
!pwd

!git branch

!git status

/content/ATM-JAVA-/ATM-JAVA-
* master
On branch master
Your branch is up to date with 'origin/master'.

nothing to commit, working tree clean


!git checkout -b Task1


In [20]:
!git checkout -b Task1

fatal: A branch named 'Task1' already exists.


In [21]:
!git branch

* Task1
  master


In [22]:
!git push -u origin Task1

fatal: could not read Username for 'https://github.com': No such device or address


In [8]:
# ── Step 1: Install PySpark ──────────────────────────────────────────────────
!pip install pyspark -q

In [9]:
# ── Step 2: Download dataset if not already present ─────────────────────────
import urllib.request, os

DATA_URL  = "https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2024-01.parquet"
DATA_PATH = "/content/fhvhv_tripdata_2024-01.parquet"

if not os.path.exists(DATA_PATH):
    print("Downloading NYC TLC HVFHV dataset (~451 MB) …")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print("Download complete.")
else:
    print(f"Dataset already present at {DATA_PATH}")

print(f"File size: {os.path.getsize(DATA_PATH)/1e6:.1f} MB")

Download complete.
File size: 472.8 MB


## 1.1 Problem Definition
This project addresses a **binary classification problem**: predicting whether a passenger will tip for a High Volume For-Hire Vehicle (HVFHV) trip in New York City. The dataset is the official NYC TLC HVFHV Trip Records for January 2024.

In [10]:
# ── Step 3: Start SparkSession ───────────────────────────────────────────────
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("7006SCN_Task1") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"PySpark version: {spark.version}")

PySpark version: 4.0.3


## 1.2 Load Dataset & Verify Row / Column Count

In [11]:
import os

df = spark.read.parquet(DATA_PATH)

row_count = df.count()
col_count = len(df.columns)
file_mb   = os.path.getsize(DATA_PATH) / 1e6

print(f"Rows    : {row_count:,}")
print(f"Columns : {col_count}")
print(f"Size    : {file_mb:.2f} MB")

Rows    : 19,663,930
Columns : 24
Size    : 472.76 MB


## 1.3 Schema

In [12]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_

## 1.4 First 20 Rows

In [13]:
df.show(20, truncate=True)

+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|   request_datetime|  on_scene_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|
+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+--